<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/Faiss_Semantic_260514.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.8 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import urllib.request
import faiss
import time
import torch
from sentence_transformers import SentenceTransformer

In [11]:
df = pd.read_csv('/content/abcnews-date-text.csv',
                on_bad_lines='skip')
print(df.head())

  publish_date                                      headline_text
0     20030219  aba decides against community broadcasting lic...
1     20030219     act fire witnesses must be aware of defamation
2     20030219     a g calls for infrastructure protection summit
3     20030219           air nz staff in aust strike for pay rise
4     20030219      air nz strike to affect australian travellers


/tmp/ipykernel_2057/3494040964.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/abcnews-date-text.csv',


In [15]:
# urllib 다운로드 부분 대신 이걸로 대체
df = pd.read_csv('/content/abcnews-date-text.csv',
                 on_bad_lines='skip',
                 engine='python',
                 dtype={'publish_date': str})
data = df.headline_text.to_list()

print(data[:5])
print('총 샘플의 개수 :', len(data))

['aba decides against community broadcasting licence', 'act fire witnesses must be aware of defamation', 'a g calls for infrastructure protection summit', 'air nz staff in aust strike for pay rise', 'air nz strike to affect australian travellers']
총 샘플의 개수 : 2360395


In [16]:
model = SentenceTransformer('distilbert-base-nli-mean-tokens')

# 전체 대신 1만개만 사용 (시간 절약)
data = data[:10000]

encoded_data = model.encode(data)
print('임베딩 된 벡터 수 :', len(encoded_data))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 된 벡터 수 : 10000


In [17]:
index = faiss.IndexIDMap(faiss.IndexFlatIP(768))
index.add_with_ids(encoded_data, np.array(range(0, len(data))))

faiss.write_index(index, 'abc_news')

1줄: 768차원(SBERT 출력 크기) Faiss 인덱스 생성

2줄: 임베딩된 벡터들을 인덱스에 추가

3줄: 인덱스를 abc_news 파일로 저장

In [18]:
def search(query):
  t = time.time()
  query_vector = model.encode([query])
  k = 5
  top_k = index.search(query_vector, k)
  print('total time: {}'.format(time.time() - t))
  return [data[_id] for _id in top_k[1].tolist()[0]]

In [19]:
query = str(input())
results = search(query)

print('results :')
for result in results:
  print('|t', result)

Underwater Forest Discovered
total time: 0.06244683265686035
results :
|t portland centre moves closer to underwater display
|t scud powers through in miami
|t moya moves into miami quarters
|t boy drowns on hinterland property
|t tourist drowns on reef


1. `faiss-cpu` 설치 후 import해야 faiss 사용 가능

2. GitHub 링크 막히면 Kaggle에서 데이터 직접 다운로드

3. 108만개 데이터는 너무 많으니 1만개로 줄여서 실습

4. SBERT로 텍스트를 768차원 벡터로 임베딩

5. Faiss 인덱스에 벡터 저장 후 유사 문장 시맨틱 검색 가능